# Vegetarian Food Image Classification — Exploration

Walkthrough of the real-photo pipeline artifacts. Run cells top-to-bottom
after `python src/train.py` and `python src/evaluate.py` have produced `results/`.

In [ ]:
import json, matplotlib.pyplot as plt
from PIL import Image

stats = json.load(open('../raw/dataset_stats.json'))
split = json.load(open('../data/split_summary.json'))
metrics = json.load(open('../results/test_metrics.json'))
hist = json.load(open('../results/training_history.json'))
print('corpus:', stats['total_images'], 'images,', len(stats['classes']), 'classes,', stats['total_mb'], 'MB')
print('subset totals:', split['totals'])
print('test metrics:', {k: metrics[k] for k in ('accuracy','f1_macro','precision_macro','recall_macro')})

In [ ]:
# verification grid — one REAL photo per class
Image.open('../results/dataset_samples_grid.png')

In [ ]:
# stage 1 (frozen) -> stage 2 (fine-tune) curves
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
n1 = hist['stage1_epochs']
ax[0].plot(hist['train_loss'], 'o-', label='train')
ax[0].plot(hist['val_loss'], 's-', label='val')
ax[0].axvline(n1 - 0.5, ls='--', c='gray'); ax[0].set_title('loss'); ax[0].legend()
ax[1].plot(hist['train_acc'], 'o-', label='train')
ax[1].plot(hist['val_acc'], 's-', label='val')
ax[1].axvline(n1 - 0.5, ls='--', c='gray'); ax[1].set_title('accuracy'); ax[1].legend()
plt.show()
print('best checkpoint:', hist['best'])

In [ ]:
# per-class test accuracy (actual model, actual test images)
preds = json.load(open('../results/test_predictions.json'))
pc = {}
for r in preds['records']:
    d = pc.setdefault(r['true'], [0, 0]); d[0] += r['correct']; d[1] += 1
for k in sorted(pc):
    c, n = pc[k]
    print(f"{k:14s} {c}/{n} = {c/n:.4f}")

In [ ]:
# confusion matrix + qualitative montage
display(Image.open('../results/confusion_matrix.png'))
display(Image.open('../results/sample_predictions.png'))